In [ ]:
!pip install -U huggingface_hub
!pip install transformers --upgrade
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.1 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.1.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 22.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.1.5
    Uninstalling huggingface_hub-1.1.5:
      Successfully uninstalled huggingface_hub-1.1.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


# **IMPORT THƯ VIỆN**

In [ ]:
# Importing necessary libraries and modules
import warnings  # Import the 'warnings' module for handling warnings
warnings.filterwarnings("ignore")  # Ignore warnings during execution

import gc  # Import the 'gc' module for garbage collection
import numpy as np  # Import NumPy for numerical operations
import pandas as pd  # Import Pandas for data manipulation
import itertools  # Import 'itertools' for iterators and looping
from collections import Counter  # Import 'Counter' for counting elements
import matplotlib.pyplot as plt  # Import Matplotlib for data visualization
from sklearn.metrics import (  # Import various metrics from scikit-learn
    accuracy_score,  # For calculating accuracy
    roc_auc_score,  # For ROC AUC score
    confusion_matrix,  # For confusion matrix
    classification_report,  # For classification report
    f1_score  # For F1 score
)

# Import custom modules and classes
from imblearn.over_sampling import RandomOverSampler # import RandomOverSampler
import accelerate # Import the 'accelerate' module
# import evaluate  # Import the 'evaluate' module
from datasets import Dataset, Image, ClassLabel  # Import custom 'Dataset', 'ClassLabel', and 'Image' classes
from transformers import (  # Import various modules from the Transformers library
    TrainingArguments,  # For training arguments
    Trainer,  # For model training
    ViTImageProcessor,  # For processing image data with ViT models
    ViTForImageClassification,  # ViT model for image classification
    DefaultDataCollator  # For collating data in the default way
)
import torch  # Import PyTorch for deep learning
from torch.utils.data import DataLoader  # For creating data loaders
from torchvision.transforms import (  # Import image transformation functions
    CenterCrop,  # Center crop an image
    Compose,  # Compose multiple image transformations
    Normalize,  # Normalize image pixel values
    RandomRotation,  # Apply random rotation to images
    RandomResizedCrop,  # Crop and resize images randomly
    RandomHorizontalFlip,  # Apply random horizontal flip
    RandomAdjustSharpness,  # Adjust sharpness randomly
    Resize,  # Resize images
    ToTensor  # Convert images to PyTorch tensors
)

# **DOWNLOAD MODEL FROM HUGGING FACE**

In [ ]:
import torch
from transformers import AutoModelForImageClassification, AutoImageProcessor
model_name = "dima806/facial_emotions_image_detection"
# subfolder_name = "checkpoint-1180"

print("Đang tải mô hình, trọng số (model.safetensors) và cấu hình...")
try:
    # Thư viện sẽ tự động tải các tệp config, safetensors và preprocessor config
    # Tải mô hình và cấu hình từ thư mục con 'checkpoint-1180'
    model = AutoModelForImageClassification.from_pretrained(
    model_name
    # subfolder=subfolder_name  # Chỉ định thư mục con
    )
    processor = AutoImageProcessor.from_pretrained(
        model_name,
        # subfolder=subfolder_name # Cần chỉ định cho cả processor
    )

    # print(f"✅ Đã tải mô hình từ thư mục con: {subfolder_name}")

except Exception as e:
    print(f"Lỗi khi tải bằng transformers: {e}")
    # Nếu thất bại, bạn sẽ phải dùng phương pháp thủ công, tải từng tệp (.safetensors, config.json, v.v.)
    # và tự xây dựng lại kiến trúc như hướng dẫn trước.

Đang tải mô hình, trọng số (model.safetensors) và cấu hình...


config.json:   0%|          | 0.00/907 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Lỗi khi tải bằng transformers: name 'subfolder_name' is not defined


# **DOWNLOAD DATASET TO EVALUATE MODEL**

## Set up Kaggle API in Colab

Follow these steps to enable Kaggle dataset downloads in your Colab environment:

1.  **Get your Kaggle API Token:**
    *   Go to [Kaggle.com](https://www.kaggle.com/), log in, and navigate to your user profile.
    *   Click on 'Account'.
    *   Scroll down to the 'API' section and click 'Create New API Token'. This will download a `kaggle.json` file.
2.  **Upload `kaggle.json` to Colab:**
    *   Run the code cell below, and a file upload dialog will appear. Upload the `kaggle.json` file you just downloaded.

In [ ]:
# Install the Kaggle API client
!pip install -q kaggle
# Create a directory for Kaggle configuration
!mkdir -p ~/.kaggle
# Upload your Kaggle API key (kaggle.json)
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"anhayng","key":"272a2db81608446b482632e8cbb25506"}'}

In [ ]:
# Move the kaggle.json file to the .kaggle directory and set permissions
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## FER2013

FER2013 (Facial Expression Recognition 2013)
- Đặc điểm: Đây là một bộ dữ liệu lớn và phổ biến, được tạo ra cho một cuộc thi Kaggle vào năm 2013.Kích thước: Hơn 35.000 hình ảnh khuôn mặt thang độ xám (grayscale) với độ phân giải thấp ($48 \times 48$ pixels).
- Các lớp cảm xúc: 7 lớp cơ bản (Tức giận, Ghê tởm, Sợ hãi, Hạnh phúc, Buồn bã, Bất ngờ, Trung tính).
- Giá trị khoa học: Mặc dù được thu thập tự động (có nhiều nhiễu), FER2013 là một thách thức lớn và được sử dụng rộng rãi để kiểm tra khả năng khái quát hóa (generalization) của mô hình

In [ ]:
# # Download the dataset from Kaggle
# !kaggle datasets download msambare/fer2013
# # Unzip the dataset
# # The dataset typically gets downloaded as a zip file in the current working directory
# !unzip -q fer2013.zip -d fer2013
# print("Dataset downloaded and unzipped successfully!")

## CK+

CK+ (Extended Cohn-Kanade Dataset)
- Đặc điểm: Đây là một trong những bộ dữ liệu được gán nhãn chính xác nhất và được sử dụng để đánh giá các mô hình ban đầu.

- Kích thước: Chứa các chuỗi hình ảnh từ trạng thái trung tính đến cảm xúc cực đại.

- Giá trị khoa học: CK+ rất quan trọng vì nó cung cấp các nhãn Action Units (AUs) (Đơn vị Hành động) theo hệ thống mã hóa hành động khuôn mặt FACS (Facial Action Coding System), cho phép đánh giá chi tiết hơn về cơ học khuôn mặt.

In [ ]:
# # Download the dataset from Kaggle
# !kaggle datasets download shuvoalok/ck-dataset

# # Unzip the dataset
# # The dataset typically gets downloaded as a zip file in the current working directory
# !unzip -q ck-dataset.zip -d ck

# print("Dataset CK downloaded and unzipped successfully!")

## AffectNet

- Đặc điểm: Một trong những bộ dữ liệu lớn nhất cho FER trong môi trường thực tế (in-the-wild).

- Kích thước: Hơn 13.000 hình ảnh được gán nhãn cảm xúc thủ công, thu thập từ các công cụ tìm kiếm trên Internet.

- Các lớp cảm xúc: Hỗ trợ 8 lớp cơ bản (7 lớp của FER2013 + Khinh miệt) và cả các thuộc tính cảm xúc liên tục (valence và arousal).

- Giá trị khoa học: AffectNet là tiêu chuẩn vàng hiện nay để đánh giá hiệu suất mô hình trong các điều kiện thực tế (ánh sáng, góc chụp, che khuất, v.v. đa dạng).

In [ ]:
# # Download the dataset from Kaggle
# !kaggle datasets download mstjebashazida/affectnet

# # Unzip the dataset
# # The dataset typically gets downloaded as a zip file in the current working directory
# !unzip -q affectnet.zip -d affectnet

# print("Dataset CK downloaded and unzipped successfully!")

Dataset URL: https://www.kaggle.com/datasets/mstjebashazida/affectnet
License(s): MIT
affectnet.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset CK downloaded and unzipped successfully!


## RAF-DB

RAF-DB (Real-world Affective Face Database)
- Đặc điểm: Tập trung vào các biểu cảm khuôn mặt trong thế giới thực với sự đa dạng cao.

- Kích thước: Khoảng 3000 hình ảnh.

- Các lớp cảm xúc: Cả 7 lớp cơ bản và các lớp cảm xúc phức tạp/hỗn hợp (compound emotions).

- Giá trị khoa học: RAF-DB là benchmark quan trọng để đo lường khả năng xử lý các biểu cảm tự nhiên và phức tạp hơn, vốn là thách thức lớn đối với các mô hình FER.

In [ ]:
# Download the dataset from Kaggle
!kaggle datasets download shuvoalok/raf-db-dataset

# Unzip the dataset
# The dataset typically gets downloaded as a zip file in the current working directory
!unzip -q raf-db-dataset.zip -d raf-db

print("Dataset RAF-DB downloaded and unzipped successfully!")

Dataset URL: https://www.kaggle.com/datasets/shuvoalok/raf-db-dataset
License(s): other
  0% 0.00/37.7M [00:00<?, ?B/s]
100% 37.7M/37.7M [00:00<00:00, 1.05GB/s]
Dataset CK downloaded and unzipped successfully!


# Xử lí dữ liệu

## Chuẩn bị Cấu trúc Dữ liệu

In [ ]:
import os
def rename_directory_os(old_directory_path, new_directory_path):
    try:
        os.rename(old_directory_path, new_directory_path)
        print(f"Directory '{old_directory_path}' renamed to '{new_directory_path}' successfully.")
    except FileNotFoundError:
        print(f"Error: Directory '{old_directory_path}' not found.")
    except OSError as e:
        print(f"Error renaming directory: {e}")

In [ ]:
rename_directory_os("/content/raf-db/DATASET/test/1", "/content/raf-db/DATASET/test/surprise")
rename_directory_os("/content/raf-db/DATASET/test/2", "/content/raf-db/DATASET/test/fear")
rename_directory_os("/content/raf-db/DATASET/test/3", "/content/raf-db/DATASET/test/disgust")
rename_directory_os("/content/raf-db/DATASET/test/4", "/content/raf-db/DATASET/test/happy")
rename_directory_os("/content/raf-db/DATASET/test/5", "/content/raf-db/DATASET/test/sad")
rename_directory_os("/content/raf-db/DATASET/test/6", "/content/raf-db/DATASET/test/angry")
rename_directory_os("/content/raf-db/DATASET/test/7", "/content/raf-db/DATASET/test/neutral")

Directory '/content/raf-db/DATASET/test/1' renamed to '/content/raf-db/DATASET/test/surprise' successfully.
Directory '/content/raf-db/DATASET/test/2' renamed to '/content/raf-db/DATASET/test/fear' successfully.
Directory '/content/raf-db/DATASET/test/3' renamed to '/content/raf-db/DATASET/test/disgust' successfully.
Directory '/content/raf-db/DATASET/test/4' renamed to '/content/raf-db/DATASET/test/happy' successfully.
Directory '/content/raf-db/DATASET/test/5' renamed to '/content/raf-db/DATASET/test/sad' successfully.
Directory '/content/raf-db/DATASET/test/6' renamed to '/content/raf-db/DATASET/test/angry' successfully.
Directory '/content/raf-db/DATASET/test/7' renamed to '/content/raf-db/DATASET/test/neutral' successfully.


In [ ]:
# Import necessary libraries
image_dict = {}

# Define the list of file names
from pathlib import Path
from tqdm import tqdm
import os
# Initialize empty lists to store file names and labels
file_names = []
labels = []

# Iterate through all image files in the specified directory
# Updated path to reflect where the dataset was unzipped in Colab
dataset_path = '/content/raf-db/DATASET/test'
for file in sorted((Path(dataset_path).glob('*/*.*'))):
    # check number of such files in a directory
    sample_dir = '/'.join(str(file).split('/')[:-1])+'/'
    label = str(file).split('/')[-2]  # Extract the label from the file path
    labels.append(label)  # Add the label to the list
    file_names.append(str(file))  # Add the file path to the list

# Print the total number of file names and labels
print(len(file_names), len(labels))

# Create a pandas dataframe from the collected file names and labels
df = pd.DataFrame.from_dict({"image": file_names, "label": labels})
print(df.shape)

3068 3068
(3068, 2)


In [ ]:
df.head()

,image,label
0,/content/raf-db/DATASET/test/angry/test_0017_a...,angry
1,/content/raf-db/DATASET/test/angry/test_0027_a...,angry
2,/content/raf-db/DATASET/test/angry/test_0037_a...,angry
3,/content/raf-db/DATASET/test/angry/test_0042_a...,angry
4,/content/raf-db/DATASET/test/angry/test_0057_a...,angry


## **Xử lý dữ liệu để đánh giá**

Chúng ta cần định nghĩa một hàm để tiền xử lý hình ảnh sao cho phù hợp với đầu vào của mô hình. Hàm này sẽ áp dụng các phép biến đổi (như đổi kích thước, chuẩn hóa) bằng `processor` đã tải.

In [ ]:
from datasets import Dataset, Image, ClassLabel
dataset = Dataset.from_pandas(df).cast_column("image", Image())

In [ ]:
# Create a list of unique labels by converting 'labels' to a set and then back to a list
labels_list = ['sad', 'disgust', 'angry', 'neutral', 'fear', 'surprise', 'happy'] # list(set(labels))

# Initialize empty dictionaries to map labels to IDs and vice versa
label2id, id2label = dict(), dict()

# Iterate over the unique labels and assign each label an ID, and vice versa
for i, label in enumerate(labels_list):
    label2id[label] = i  # Map the label to its corresponding ID
    id2label[i] = label  # Map the ID to its corresponding label

# Print the resulting dictionaries for reference
print("Mapping of IDs to Labels:", id2label, '\n')
print("Mapping of Labels to IDs:", label2id)

Mapping of IDs to Labels: {0: 'sad', 1: 'disgust', 2: 'angry', 3: 'neutral', 4: 'fear', 5: 'surprise', 6: 'happy'} 

Mapping of Labels to IDs: {'sad': 0, 'disgust': 1, 'angry': 2, 'neutral': 3, 'fear': 4, 'surprise': 5, 'happy': 6}


In [ ]:
from datasets import ClassLabel

# Convert label column to ClassLabel feature
dataset = dataset.cast_column("label", ClassLabel(names=labels_list))

def preprocess_images(examples):
    """Preprocesses image examples using the loaded processor."""
    # Ensure image is in RGB format if it's grayscale
    images = [image.convert("RGB") for image in examples["image"]]
    # Apply the image processor's transformations
    examples["pixel_values"] = processor(images, return_tensors="pt").pixel_values
    return examples

# Apply preprocessing to the dataset
processed_dataset = dataset.map(preprocess_images, batched=True)

# Rename the 'label' column to 'labels' as expected by the Trainer
processed_dataset = processed_dataset.rename_column("label", "labels")

# Set the format to PyTorch tensors
processed_dataset.set_format("torch")

print("Dữ liệu đã được tiền xử lý và sẵn sàng để đánh giá.")

Casting the dataset:   0%|          | 0/3068 [00:00<?, ? examples/s]

Map:   0%|          | 0/3068 [00:00<?, ? examples/s]

Dữ liệu đã được tiền xử lý và sẵn sàng để đánh giá.


## **Thiết lập Đánh giá Mô hình**

Để đánh giá mô hình, chúng ta sẽ sử dụng thư viện `evaluate` của Hugging Face để tính toán các chỉ số như độ chính xác (accuracy) và F1-score. Sau đó, chúng ta sẽ định nghĩa một hàm `compute_metrics` để tính toán các chỉ số này trong quá trình đánh giá.

In [ ]:
import evaluate

# Load the accuracy metric
accuracy_metric = evaluate.load("accuracy")

# Load the f1 metric
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    """Computes accuracy and f1-score for evaluation predictions."""
    predictions, labels = eval_pred
    # Get the predicted labels (argmax of logits)
    predicted_labels = np.argmax(predictions, axis=1)

    # Compute accuracy
    accuracy = accuracy_metric.compute(predictions=predicted_labels, references=labels)

    # Compute F1 score (weighted average, as it's a multi-class problem)
    f1 = f1_metric.compute(predictions=predicted_labels, references=labels, average="weighted")

    return {**accuracy, **f1}

print("Hàm `compute_metrics` đã được định nghĩa.")

Hàm `compute_metrics` đã được định nghĩa.


## **Chạy Đánh giá**

Bây giờ chúng ta sẽ thiết lập `TrainingArguments` (dù chỉ để đánh giá) và `Trainer`, sau đó chạy phương thức `evaluate` để nhận kết quả.

In [ ]:
import evaluate
import numpy as np
import pandas as pd # Import pandas for DataFrame operations
import os # Import os for path manipulation

# Define training arguments for evaluation
eval_args = TrainingArguments(
    output_dir="./evaluation_results",  # Directory for storing evaluation results
    per_device_eval_batch_size=16,       # Batch size per device for evaluation (giảm từ 32 xuống 16)
    do_eval=True,                        # Perform evaluation
    remove_unused_columns=False,         # Keep unused columns in the dataset
    report_to="none"                     # Do not report to any external service
)

# Create a Trainer instance for evaluation
trainer = Trainer(
    model=model,                         # The model to evaluate
    args=eval_args,                      # Evaluation arguments
    eval_dataset=processed_dataset,      # The dataset to evaluate on
    compute_metrics=compute_metrics,     # Function to compute metrics
    tokenizer=processor,                 # The processor (tokenizer is also used for image tasks)
)

# Run the evaluation (for overall metrics)
evaluation_results = trainer.evaluate()

print("Kết quả đánh giá:")
print(evaluation_results)

Kết quả đánh giá:
{'eval_loss': 1.2839879989624023, 'eval_model_preparation_time': 0.0125, 'eval_accuracy': 0.5798565840938722, 'eval_f1': 0.5956547620063429, 'eval_runtime': 2093.738, 'eval_samples_per_second': 1.465, 'eval_steps_per_second': 0.092}

--- Bắt đầu phân tích dự đoán sai ---


KeyboardInterrupt: 

In [ ]:
import pandas as pd

# Convert the evaluation_results dictionary to a DataFrame
# We'll create a single-row DataFrame
eval_df = pd.DataFrame([evaluation_results])

# Display the DataFrame to verify
display(eval_df)


,eval_loss,eval_model_preparation_time,eval_accuracy,eval_f1,eval_runtime,eval_samples_per_second,eval_steps_per_second
0,1.283988,0.0125,0.579857,0.595655,2093.738,1.465,0.092


In [ ]:
# Save the DataFrame to a CSV file
csv_filename = "evaluation_results_RAF-DB.csv"
eval_df.to_csv(csv_filename, index=False)

from google.colab import files
# Assuming the csv_filename variable holds the name of your CSV file
files.download(csv_filename)
print(f"Kết quả đánh giá đã được lưu vào tệp: {csv_filename}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Kết quả đánh giá đã được lưu vào tệp: evaluation_results_RAF-DB.csv
